# GSB 5544 — Data Visualization: Asking Questions with Plots  
*Sections 0–3 are fully worked. From Section 4 on, fill in every `____` blank; from Section 8 on, write the code yourself.*

Every section starts with a **question**; the plot is just how we answer it. Pick the plot from the question and the *types of the variables* — the code follows.

**Data:** `kobe_bryant_games.csv` — every regular-season game Kobe Bryant played in his first 15 NBA seasons (1996–2011). One **row** = one game.

| Column | Meaning | Type |
|---|---|---|
| `season`, `season_label` | 1–15 and e.g. `"2005-06"` | categorical (ordered) |
| `date` | game date | date |
| `location` | `Home` / `Away` | categorical |
| `outcome` | `W` / `L` for the Lakers | categorical |
| `point_margin` | Lakers score − opponent score | quantitative |
| `opponent`, `opp_conference` | opponent, `Eastern` / `Western` | categorical |
| `started` | 1 = started, 0 = came off the bench | categorical (stored as a number!) |
| `minutes`, `points`, `rebounds`, `assists`, `steals`, `blocks`, `turnovers` | box-score stats | quantitative |
| `fg`, `fga`, `fg_pct`, `fg3`, `fg3a`, `ft`, `fta` | shots made / attempted | quantitative |
| `game_score` | Hollinger's single-game productivity score | quantitative |
| `double_double`, `triple_double` | 1 = yes, 0 = no | categorical |
| `age` | Kobe's age (years) | quantitative |

Textbook: [Chapter 3 — Data Visualization in Python](https://ds-ml-with-python.github.io/Course-Textbook/02-plotnine.html)


---
## What Did We Learn Last Week?

Two practice activities, and every tool below comes back today:

**PA 1-1 (Decode the Message):** Python objects are built and combined — **lists**, **NumPy arrays**, and vectorized math (`array * 2` transforms every element at once, no loop). We `import` libraries to get new tools, and we assembled our first **DataFrame** from arrays.

**PA 1-2 (Intro to Pandas):** A table comes in with `pd.read_csv(url)`. Every column is a variable, and **you decide its type** — quantitative, categorical, or date/time — regardless of how pandas stored it:

| Last week's tool | What it did | Where you need it today |
|---|---|---|
| `pd.read_csv()` | load a CSV into a DataFrame | every data set today |
| `.dtypes`, `type()` | see how pandas stored each column | checking your conversions |
| `.astype(str)` / `.astype("category")` | *declare* a variable categorical | PA 2.1 asks you to convert `cyl`, `am`, ... before plotting |
| `df["col"]`, `df[["a","b"]]`, `.loc` | grab columns and rows | picking what to plot |
| `.map({...})`, `pd.cut()` | recode / bin into categories | choosing what counts as a "group" in a plot |

The one-line summary: **the type of the variable decides the plot.** A number that is really a label (like `cyl` = 4/6/8) must be converted *before* plotting, or today's plots will come out wrong — that is literally questions 1–4 of Practice Activity 2.1.

### ⏱️ Warm-up — fill these in as we review (last week's coffee data)

In [ ]:
import ______ as pd

coffee = pd.________("https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/coffee_purchases.csv")
coffee.______()        # peek at the first few rows

In [ ]:
# Which columns does pandas THINK are numbers, but are really categorical?
coffee.______

# Declare them -- exactly what you'll do to cyl and am today
coffee["account"] = coffee["account"].astype("________")
coffee["year"]    = coffee["year"].astype(___)
coffee.dtypes

---
## 0. *What actually happens when I "load a package"?*  → `import`

Python starts almost empty. `pandas` and `plotnine` are **libraries**: folders of code other people wrote and shared. Nothing in them exists in your session until you `import` it.

| You write | What happens |
|---|---|
| `import pandas as pd` | load the whole `pandas` library and give it the nickname `pd`; you reach inside with `pd.read_csv(...)` |
| `from plotnine import ggplot, aes` | reach into `plotnine` and pull out **just those names**; you can now type `ggplot(...)` directly |
| `from plotnine import *` | pull out **everything** — convenient, but now you can't tell where a name came from |

`import` only *loads*; **installing** (downloading the library to your machine) happens once, with `pip`. Colab already has `pandas`; it does **not** have `plotnine`, so the first line below installs it.


In [ ]:
# Install once per machine / Colab session (the ! sends the line to the terminal, not Python)
!pip install plotnine -q


In [ ]:
# Q: Is `ggplot` a thing yet?  Not until we import it.
try:
    ggplot
except NameError:
    print("NameError: ggplot does not exist in this session yet")


In [ ]:
import pandas as pd                       # the library, nicknamed pd
from plotnine import *                    # every plotting function in plotnine: ggplot, aes, geom_*, ...

print(type(pd))                           # pd is a *module* (a loaded library)
print(pd.__version__)                     # which version got loaded
ggplot                                    # now it exists


In [ ]:
kobe = pd.read_csv("https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/kobe_bryant_games.csv")
kobe.head()


---
## 1. *Why draw a picture at all?*

**Q: What does a typical Kobe game look like — how many points does he score?**

We can *answer* with numbers…


In [ ]:
kobe["points"].describe()


…and that's a fine answer. But the mean hides whether 25 points is "most nights" or "an average of 10-point nights and 40-point nights." A picture shows the *whole* distribution at once:


In [ ]:
(ggplot(kobe, aes(x = "points"))
 + geom_histogram(bins = 25)
)


Now we can see the shape: most games are 20–35 points, a long tail of 40+, and a lump near zero (rookie-year bench minutes). **A plot answers a question that a summary number can't.**

### The type of the variable decides the plot

Before touching code, ask: *what kind of variable(s) am I plotting?* (Same definitions as last week.)

| Question involves | Plot | plotnine `geom_*` |
|---|---|---|
| one **categorical** variable | bar plot (counts) | `geom_bar` |
| one **quantitative** variable | histogram / density | `geom_histogram`, `geom_density` |
| a **quantitative** by a **categorical** | side-by-side boxplots | `geom_boxplot` |
| two **quantitative** variables | scatterplot | `geom_point` |
| a **quantitative** over **time / order** | line plot | `geom_line` |
| two **categorical** variables | stacked / dodged bars | `geom_bar` + `fill` |

Remember: `started` is stored as `0`/`1` and `season` as `1`–`15`, but they are **categorical**. The dtype does not decide — you do.


---
## 2. *How do I describe a plot to the computer?*  → the grammar of graphics: `ggplot()` + `aes()` + `geom_*()`

**Q: How many games did the Lakers win vs. lose with Kobe on the floor?**

One categorical variable (`outcome`) → a **bar plot**. Every `plotnine` plot is the *same three ingredients* added together with `+`:

1. **the data** — `ggplot(kobe, ...)`
2. **the aesthetics** — `aes(x = "outcome")`: *which column* controls *which visual element* (x position, y position, color, fill, size…)
3. **the geometry** — `geom_bar()`: *what shape* to draw

The parentheses around the whole thing let us split it across lines.


In [ ]:
(ggplot(kobe, aes(x = "outcome"))     # data + aesthetic: outcome on the x-axis
 + geom_bar()                         # geometry: a bar whose height is the COUNT of rows
)


`geom_bar()` counted the rows for us — 733 wins, 370 losses. We never computed a count; the geometry did.

### Aesthetic vs. constant

**Q: Can I color the bars?** Yes — two very different ways:

| Where you put it | Meaning |
|---|---|
| inside `aes(fill = "outcome")` | *map* a **column** to fill → each level gets its own color and a legend |
| outside: `geom_bar(fill = "purple")` | *set* a **constant** → every bar is purple, no legend |


In [ ]:
# fill MAPPED to a variable -> a legend appears
(ggplot(kobe, aes(x = "outcome", fill = "outcome"))
 + geom_bar()
)


In [ ]:
# fill SET to a constant -> no legend
(ggplot(kobe, aes(x = "outcome"))
 + geom_bar(fill = "purple")
)


✅ **Check:** predict what `aes(x = "outcome", fill = "purple")` does *before* running it. (Hint: `aes` treats everything inside it as data.)


In [ ]:
# Predict first, THEN run:
(ggplot(kobe, aes(x = "outcome", fill = "purple"))
 + geom_bar()
)

---
## 3. *Does the answer depend on a second categorical variable?*  → `fill` + `position`

**Q: Does Kobe win more at home than on the road?**

Two categorical variables (`outcome` and `location`). Put one on `x` and map the other to `fill`.


In [ ]:
(ggplot(kobe, aes(x = "location", fill = "outcome"))
 + geom_bar()
)


Stacked bars show *totals* but make the win share hard to compare. `position` changes how the bars are arranged:

| `position=` | Shows |
|---|---|
| `"stack"` (default) | counts, stacked |
| `"dodge"` | counts, side by side |
| `"fill"` | **proportions** — every bar stretched to 1 |


In [ ]:
(ggplot(kobe, aes(x = "location", fill = "outcome"))
 + geom_bar(position = "dodge")
)


In [ ]:
(ggplot(kobe, aes(x = "location", fill = "outcome"))
 + geom_bar(position = "fill")
)


The `"fill"` version answers the question directly: the win *share* is much higher at home (about 76%) than away (about 57%). Same data, three pictures — **the question decides the position.**

✅ **Check:** Is Kobe more likely to record a double-double when he *starts* than when he comes off the bench? `started` and `double_double` are numbers — the plot will treat them as quantitative unless you convert them. Convert both to `category`, then plot with `position = "fill"`.


In [ ]:
# double-doubles at home vs away -- which position= answers a SHARE question?
(ggplot(kobe, aes(x = "____", fill = "____"))
 + geom_bar(position = "____")
)

---
## 4. *What is the shape of a quantitative variable?*  → `geom_histogram`, `geom_density`

**Q: How many minutes does Kobe play in a game?**

One quantitative variable → a **histogram**. The only decision is how many bins: too few hides the shape, too many shows noise.

*Fill in the blanks so the cell answers the question in its comment.*


In [ ]:
# Q: Minutes per game - try 10, 30, and 60 bins. Which tells the truest story?
(ggplot(kobe, aes(x = "____"))
 + geom_histogram(bins = ____)
)


**Q: Does his scoring distribution look different at home vs. away?**

Two overlapping histograms are hard to read. A **density** is a smoothed histogram; with `alpha` (transparency) two of them can share a plot.


In [ ]:
(ggplot(kobe, aes(x = "____", fill = "____"))
 + geom_density(alpha = ____)       # alpha = transparency, 0 (invisible) to 1 (solid)
)


The two curves sit almost on top of each other — home court barely changes *how much* he scores, even though it changes *whether they win* (Section 3).

✅ **Check:** compare the distribution of `game_score` for wins vs. losses with a density plot. Which way does it shift, and by roughly how much?


In [ ]:
# game_score for wins vs losses
(ggplot(kobe, aes(x = "____", fill = "____"))
 + geom_density(alpha = ____)
)

---
## 5. *How does a quantitative variable differ across groups?*  → `geom_boxplot`

**Q: Which of Kobe's seasons was his highest scoring?**

A quantitative variable (`points`) split by a categorical one (`season_label`) → **side-by-side boxplots**. Each box is one season: median line, middle-50% box, whiskers, and outlier dots.


In [ ]:
(ggplot(kobe, aes(x = "____", y = "____"))
 + geom_____()
)


2005-06 stands out (median = 36 — the 81-point game is the dot at the top). But the x-axis labels are crushed together; we'll fix that in Section 8.

✅ **Check:** does Kobe score more against the Eastern or Western conference? Boxplots of `points` by `opp_conference`, and color the boxes by conference.


In [ ]:
# points by opponent conference
(ggplot(kobe, aes(x = "____", y = "____"))
 + geom_________()
)

---
## 6. *Are two quantitative variables related?*  → `geom_point`

**Q: Do more shot attempts mean more points?**

Two quantitative variables → a **scatterplot**: one dot per game.


In [ ]:
(ggplot(kobe, aes(x = "____", y = "____"))
 + geom_point()
)


Strong upward trend, but wide: 25 attempts can be 20 points or 45. A third variable can ride along as `color`:


In [ ]:
# Q: Are the high-attempt games more often wins or losses?
(ggplot(kobe, aes(x = "fga", y = "points", color = "____"))
 + geom_point(alpha = 0.5)
)


(`color` is for dots and lines; `fill` is for bars, boxes, and densities.)

✅ **Check:** is there a relationship between `assists` and `turnovers`? Many games share the exact same (assists, turnovers) pair, so the dots stack — try `geom_jitter()` instead of `geom_point()`.


In [ ]:
# assists vs turnovers -- the dots stack, which geom un-stacks them?
(ggplot(kobe, aes(x = "____", y = "____"))
 + geom_______()
)

---
## 7. *How does something change over time?*  → `geom_line`

**Q: How did Kobe's scoring average change across his career?**

A line plot needs **one y-value per x-value**. We have 82 games per season, so we first *summarize* (average points per season — next week's topic, but you saw `.groupby` in the book) and then draw the line.


In [ ]:
# Step 1: one row per season
by_season = kobe.groupby("____")["____"].mean().reset_index()
by_season.head()


In [ ]:
# Step 2: line + points   (season is numeric here, so the x-axis is in order automatically)
(ggplot(____, aes(x = "____", y = "____"))
 + geom_____()
 + geom_point()
)


A career arc in one picture: bench rookie, rising through 2005-06, then a slow decline.

✅ **Check:** compute the average `points` per season **separately for home and away** (`groupby(["season", "location"])`), then draw two lines with `color = "location"`. Why does `geom_line` need `color` (or `group`) here?


In [ ]:
# average points per season, split by location
by_season_loc = kobe.groupby(["____", "____"])["____"].mean().reset_index()

(ggplot(by_season_loc, aes(x = "____", y = "____", color = "____"))
 + geom_____()
 + geom_point()
)

---
## 8. *How do I make the plot presentable?*  → `labs()`, `scale_*`, `theme_*`

Everything so far answered the question *for us*. To show it to someone else we need labels, sensible axes, and a clean look. These are extra layers — just keep adding with `+`.

| Layer | Controls |
|---|---|
| `labs(title=, x=, y=, fill=, color=)` | text on the plot |
| `scale_x_continuous(breaks=...)`, `scale_y_reverse()`, `scale_x_log10()` | how a **numeric** axis is drawn |
| `scale_fill_manual(values=[...])`, `scale_color_manual(...)` | which colors the levels get |
| `theme_minimal()`, `theme_classic()`, `theme_bw()` | overall look |
| `theme(axis_text_x = element_text(angle = 45, hjust = 1))` | one specific element |

Start from the Section 5 boxplot and fix the crushed labels:


In [ ]:
(ggplot(kobe, aes(x = "season_label", y = "points"))
 + geom_boxplot(fill = "gold")
 + labs(title = "____",
        x = "____", y = "____")
 + theme_minimal()
 + theme(axis_text_x = element_text(angle = ____, hjust = 1))
)


Now the career-average line with honest axis ticks and Lakers colors:


In [ ]:
(ggplot(by_season_loc, aes(x = "season", y = "points", color = "location"))
 + geom_line()
 + geom_point()
 + scale_x_continuous(breaks = ____)                          # a tick at every season, 1..15
 + scale_color_manual(values = ["#552583", "#FDB927"])        # Lakers purple & gold
 + labs(title = "____", x = "____", y = "____", color = "____")
 + theme_classic()
)


✅ **Check:** take the `position = "fill"` bar plot from Section 3 and (a) label the y-axis "Share of games", (b) title it, (c) color wins gold and losses purple, (d) use `theme_bw()`.


In [ ]:
# polish the Section 3 plot
(ggplot(kobe, aes(x = "location", fill = "outcome"))
 + geom_bar(position = "____")
 + labs(title = "____", y = "____")
 + scale_fill_manual(values = ["____", "____"])
 + theme____()
)

---
## 9. *Does the pattern hold within every group?*  → `facet_wrap()`

**Q: Does the shots→points relationship look the same in wins and losses?**

Color (Section 6) put both groups on one panel. `facet_wrap("variable")` draws **one panel per level** — same axes, so panels compare directly.


In [ ]:
(ggplot(kobe, aes(x = "fga", y = "points"))
 + geom_point(alpha = 0.4)
 + facet_wrap("____")
)


Same slope in both panels — a loss is not "Kobe shooting worse", there are just fewer of them. Facets can take a second variable too:


In [ ]:
# two facet variables: rows ~ columns
(ggplot(kobe, aes(x = "fga", y = "points", color = "outcome"))
 + geom_point(alpha = 0.4)
 + facet_grid("____ ~ ____")
)

✅ **Check:** the Section 4 density of `points` by `location` was hard to read for a single season. Draw `geom_density(alpha = 0.5)` of `points` filled by `location`, faceted by `season_label`. Which seasons show a real home/away gap?


In [ ]:
# points density by location, one panel per season
(ggplot(kobe, aes(x = "____", fill = "____"))
 + geom_density(alpha = 0.5)
 + facet_wrap("____")
)

---
## 10. Practice Activities — you pick the plot

For each question: (1) name the variable types, (2) pick the geometry from the table in Section 1, (3) build the plot, (4) answer in one sentence. Solutions will be posted after class.

**PA 1.** How many games did Kobe play against each conference, and how did they split into wins and losses?


In [ ]:
# PA 1: variable types: ____ and ____ -> geometry: geom_____
(ggplot(kobe, aes(x = "____", fill = "____"))
 + geom_____(position = "____")
)

**PA 2.** How does the distribution of `minutes` compare between games Kobe started and games he came off the bench?

In [ ]:
# PA 2: variable types: ____ -> geometry: geom_____
(ggplot(kobe, aes(x = "____", y = "____"))
 + geom_________()
)

**PA 3.** Is there a relationship between `point_margin` (how much the Lakers won or lost by) and Kobe's `game_score`? Color by `outcome`. Does anything about the colors look "too obvious", and why?

In [ ]:
# PA 3
(ggplot(kobe, aes(x = "____", y = "____", color = "____"))
 + geom_______(alpha = 0.5)
)

**PA 4.** How did Kobe's average `fg_pct` change over his career? Draw one line per `location`, label the axes, and use a theme.

In [ ]:
# PA 4: summarize first, then draw
fg_by_season = kobe.groupby(["____", "____"])["____"].mean().reset_index()

(ggplot(fg_by_season, aes(x = "____", y = "____", color = "____"))
 + geom_____()
 + geom_point()
)

**PA 5.** Recreate this plot from its *grammar* description only:

- data: `kobe`; x = `rebounds`, fill = `location`
- geometry: density with `alpha = 0.4`
- one panel per `outcome`
- title "Rebounds per game", theme `theme_bw()`


In [ ]:
# PA 5: build it straight from the grammar description
(ggplot(kobe, aes(x = "____", fill = "____"))
 + geom_________(alpha = ____)
 + facet_wrap("____")
 + labs(title = "____")
 + theme____()
)

**PA 6 (write, don't code).** A classmate makes a scatterplot with `x = "season_label"` and `y = "points"` — one dot per game — and says it shows the career trend. Name two reasons a boxplot or a summarized line plot answers that question better.


---
## Summary

| Question | Variable types | Tool |
|---|---|---|
| What happens when I load a package? | — | `import pandas as pd`, `from plotnine import *` |
| How do I describe a plot? | — | `ggplot(data, aes(...)) + geom_*()` |
| Column → visual element, or a constant? | — | inside `aes(fill="col")` vs. outside `geom_bar(fill="purple")` |
| How many in each group? | 1 categorical | `geom_bar()` |
| …split by a second group? | 2 categorical | `aes(fill=)` + `position = "stack" / "dodge" / "fill"` |
| What shape is this variable? | 1 quantitative | `geom_histogram(bins=)`, `geom_density(alpha=)` |
| How does it differ across groups? | quantitative × categorical | `geom_boxplot()` |
| Are two variables related? | 2 quantitative | `geom_point()`, `geom_jitter()` |
| How does it change over time? | quantitative over order | summarize, then `geom_line()` + `geom_point()` |
| Make it presentable? | — | `labs()`, `scale_*()`, `theme_*()`, `theme(...)` |
| Does the pattern hold in every group? | + 1 categorical | `facet_wrap("col")`, `facet_grid("row ~ col")` |
